# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Baseline Rule**

I will rank content items for review using two observable signals: content staleness and CTR opportunity. Content staleness is measured by the number of days since the content was last updated. CTR opportunity is identified using search impressions, average search position, and observed click-through rate. A higher score indicates a higher priority for human review. The rule is intended for decision support and prioritization, not as proof that updating a page will improve its future performance.

**Reason Codes**
- **STALE_CONTENT**: The content has not been updated for a long period and may deserve a refresh review.
- **LOW_CTR_OPPORTUNITY**: The content receives search impressions and has a reasonable average position but has relatively low CTR.
- **STALE_AND_LOW_CTR**: The content shows both staleness and a potential CTR opportunity.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

```
dim_content(
  client_hash_id
content_hash_id
content_updated_date
is_published
is_deleted
)
        │
        │ content_hash_id + client_hash_id
        ↓
fact_content_daily_performance(
  report_date
client_hash_id
content_hash_id
gsc_data_available
gsc_impressions
gsc_clicks
gsc_avg_position
)
```

### Calculating Baseline Action Score and Rank

I will now implement the logic to calculate the `baseline_action_score` based on 'content staleness' and 'CTR opportunity', and then rank the content items.

First, I'll create sample data for `dim_content` and `fact_content_daily_performance` to demonstrate the process. In a real scenario, these would be loaded from your data sources.

In [1]:
import pandas as pd
import numpy as np
from datasets import load_dataset

# --- 1. Load Data from Hugging Face ---

print("Loading 'FlyRank/internship-warehouse' dataset configurations...")
try:
    # Load dim_content configuration
    print("Loading 'dim_content'...")
    dataset_dim_content = load_dataset('FlyRank/internship-warehouse', 'dim_content', split='train')
    df_dim_content = dataset_dim_content.to_pandas()
    print("df_dim_content loaded successfully. Head:")
    display(df_dim_content.head())

    # Load fact_content_daily_performance configuration
    print("\nLoading 'fact_content_daily_performance'...")
    dataset_fact_performance = load_dataset('FlyRank/internship-warehouse', 'fact_content_daily_performance', split='train')
    df_fact_performance = dataset_fact_performance.to_pandas()
    print("df_fact_performance loaded successfully. Head:")
    display(df_fact_performance.head())

except Exception as e:
    print(f"Error loading dataset configurations: {e}")
    print("Proceeding with synthetic data generation as a fallback.")

    # Fallback to previous synthetic data generation if Hugging Face load fails
    df_hf_sample = pd.DataFrame({'text': ['sample text ' + str(i) for i in range(100)], 'label': [i % 2 for i in range(100)]})
    df_dim_content = pd.DataFrame({
        'client_hash_id': df_hf_sample['label'].apply(lambda x: f'client_{x}'),
        'content_hash_id': ['hf_content_' + str(i) for i in range(len(df_hf_sample))],
        'content_updated_date': pd.to_datetime('2023-01-01') + pd.to_timedelta(np.arange(len(df_hf_sample)), unit='D'),
        'is_published': True,
        'is_deleted': False
    })

    num_performance_entries = len(df_dim_content) * 2
    data_fact_performance = {
        'report_date': pd.to_datetime('2024-05-20') + pd.to_timedelta(np.random.randint(0, 2, num_performance_entries), unit='D'),
        'client_hash_id': np.random.choice(df_dim_content['client_hash_id'].unique(), num_performance_entries),
        'content_hash_id': np.random.choice(df_dim_content['content_hash_id'].unique(), num_performance_entries),
        'gsc_data_available': True,
        'gsc_impressions': np.random.randint(100, 10000, num_performance_entries),
        'gsc_clicks': np.random.randint(1, 200, num_performance_entries),
        'gsc_avg_position': np.random.uniform(1.0, 50.0, num_performance_entries)
    }
    df_fact_performance = pd.DataFrame(data_fact_performance)

print("\nFinal df_dim_content head:")
display(df_dim_content.head())
print("\nFinal df_fact_performance head:")
display(df_fact_performance.head())

Loading 'FlyRank/internship-warehouse' dataset configurations...
Loading 'dim_content'...


README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

Error loading dataset configurations: Dataset 'FlyRank/internship-warehouse' is a gated dataset on the Hub. You must be authenticated to access it.
Proceeding with synthetic data generation as a fallback.

Final df_dim_content head:


,client_hash_id,content_hash_id,content_updated_date,is_published,is_deleted
0,client_0,hf_content_0,2023-01-01,True,False
1,client_1,hf_content_1,2023-01-02,True,False
2,client_0,hf_content_2,2023-01-03,True,False
3,client_1,hf_content_3,2023-01-04,True,False
4,client_0,hf_content_4,2023-01-05,True,False



Final df_fact_performance head:


,report_date,client_hash_id,content_hash_id,gsc_data_available,gsc_impressions,gsc_clicks,gsc_avg_position
0,2024-05-21,client_1,hf_content_60,True,4109,193,37.118016
1,2024-05-20,client_1,hf_content_34,True,2154,119,44.868831
2,2024-05-20,client_1,hf_content_62,True,3715,60,39.441032
3,2024-05-20,client_0,hf_content_29,True,6751,163,16.908803
4,2024-05-21,client_0,hf_content_67,True,5998,9,10.043621


In [7]:
# --- 2. Pre-process and Aggregate Performance Data ---
# Ensure report_date is datetime type
df_fact_performance['report_date'] = pd.to_datetime(df_fact_performance['report_date'])

print(f"Shape of df_dim_content before merge: {df_dim_content.shape}")
print(f"Shape of df_fact_performance before merge: {df_fact_performance.shape}")

# For each content item, get the latest report date and aggregate performance metrics
df_agg_performance = df_fact_performance.groupby(['client_hash_id', 'content_hash_id']).agg(
    latest_report_date=('report_date', 'max'),
    total_impressions=('gsc_impressions', 'sum'),
    total_clicks=('gsc_clicks', 'sum'),
    avg_position=('gsc_avg_position', 'mean'),
    gsc_data_available=('gsc_data_available', 'max') # Include gsc_data_available
).reset_index()

# --- 3. Merge DataFrames ---
df_merged = pd.merge(
    df_dim_content,
    df_agg_performance,
    on=['client_hash_id', 'content_hash_id'],
    how='left'
)

# Handle cases where content might not have performance data
# Fill numerical columns with 0 and avg_position with the mean
df_merged['total_impressions'] = df_merged['total_impressions'].fillna(0)
df_merged['total_clicks'] = df_merged['total_clicks'].fillna(0)
df_merged['avg_position'] = df_merged['avg_position'].fillna(df_merged['avg_position'].mean())

# For latest_report_date, use content_updated_date if no performance data exists, or current timestamp if content_updated_date is also missing
df_merged['latest_report_date'] = df_merged['latest_report_date'].fillna(df_merged['content_updated_date'])
df_merged['latest_report_date'] = df_merged['latest_report_date'].fillna(pd.Timestamp.now())

# Fill gsc_data_available for content without performance data (assume False if no GSC data aggregated)
df_merged['gsc_data_available'] = df_merged['gsc_data_available'].fillna(False)

print(f"Shape of df_merged after merge: {df_merged.shape}")
print("df_merged head:")
display(df_merged.head())

Shape of df_dim_content before merge: (100, 5)
Shape of df_fact_performance before merge: (200, 7)
Shape of df_merged after merge: (100, 10)
df_merged head:


/tmp/ipykernel_1280/2140520554.py:36: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_merged['gsc_data_available'] = df_merged['gsc_data_available'].fillna(False)


,client_hash_id,content_hash_id,content_updated_date,is_published,is_deleted,latest_report_date,total_impressions,total_clicks,avg_position,gsc_data_available
0,client_0,hf_content_0,2023-01-01,True,False,2024-05-20,3166.0,146.0,12.228637,True
1,client_1,hf_content_1,2023-01-02,True,False,2024-05-20,1572.0,154.0,20.210604,True
2,client_0,hf_content_2,2023-01-03,True,False,2024-05-21,9752.0,288.0,8.972813,True
3,client_1,hf_content_3,2023-01-04,True,False,2023-01-04,0.0,0.0,25.988910,False
4,client_0,hf_content_4,2023-01-05,True,False,2024-05-20,9183.0,123.0,1.544187,True


In [9]:
# --- 4. Calculate Scores ---

# Staleness Score
# Days since content was last updated relative to the latest report date for that content
df_merged['days_since_update'] = (df_merged['latest_report_date'] - df_merged['content_updated_date']).dt.days

# Normalize staleness: higher days = higher score
max_days_since_update = df_merged['days_since_update'].max()
df_merged['staleness_score'] = df_merged['days_since_update'] / max_days_since_update

# CTR Opportunity Score
# Calculate CTR, handle division by zero for impressions
df_merged['CTR'] = np.where(
    df_merged['total_impressions'] > 0,
    df_merged['total_clicks'] / df_merged['total_impressions'],
    0
)

# Normalize components for CTR opportunity
# Impressions: higher is better
max_impressions = df_merged['total_impressions'].max()
df_merged['norm_impressions'] = df_merged['total_impressions'] / max_impressions if max_impressions > 0 else 0

# Low CTR: 1 - CTR (higher if CTR is low)
df_merged['low_ctr_factor'] = 1 - df_merged['CTR']

# Avg Position: lower is better (so invert or use 1 - (pos/max_pos))
# Capped at a reasonable max position to avoid skewing by extremely high positions
max_eff_position = df_merged['avg_position'].max()
df_merged['norm_avg_position_inv'] = 1 - (df_merged['avg_position'] / max_eff_position) if max_eff_position > 0 else 0

# Combine factors for CTR opportunity score
# This formula emphasizes pages with high impressions, good position (low value), and low CTR (high 1-CTR value)
# We'll multiply these factors. If impressions are 0, this should be 0.
df_merged['ctr_opportunity_score'] = (
    df_merged['norm_impressions'] *
    df_merged['low_ctr_factor'] *
    df_merged['norm_avg_position_inv']
)

# Handle potential NaN if any normalisation had max_val = 0 for all entries (e.g. all 0 impressions)
df_merged['ctr_opportunity_score'] = df_merged['ctr_opportunity_score'].fillna(0)

# Baseline Action Score
# Weighted sum of staleness and CTR opportunity scores
# Weights can be adjusted based on business priorities
weight_staleness = 0.5
weight_ctr_opportunity = 0.5
df_merged['baseline_action_score'] = (
    (df_merged['staleness_score'] * weight_staleness) +
    (df_merged['ctr_opportunity_score'] * weight_ctr_opportunity)
)

print("DataFrame with calculated scores:")
display(df_merged[['client_hash_id', 'content_hash_id', 'content_updated_date', 'latest_report_date', 'days_since_update', 'total_impressions', 'CTR', 'staleness_score', 'ctr_opportunity_score', 'baseline_action_score']].head(10))

DataFrame with calculated scores:


,client_hash_id,content_hash_id,content_updated_date,latest_report_date,days_since_update,total_impressions,CTR,staleness_score,ctr_opportunity_score,baseline_action_score
0,client_0,hf_content_0,2023-01-01,2024-05-20,505,3166.0,0.046115,1.000000,0.110338,0.555169
1,client_1,hf_content_1,2023-01-02,2024-05-20,504,1572.0,0.097964,0.998020,0.040711,0.519365
2,client_0,hf_content_2,2023-01-03,2024-05-21,504,9752.0,0.029532,0.998020,0.375983,0.687001
3,client_1,hf_content_3,2023-01-04,2023-01-04,0,0.0,0.000000,0.000000,0.000000,0.000000
4,client_0,hf_content_4,2023-01-05,2024-05-20,501,9183.0,0.013394,0.992079,0.425919,0.708999
5,client_1,hf_content_5,2023-01-06,2023-01-06,0,0.0,0.000000,0.000000,0.000000,0.000000
6,client_0,hf_content_6,2023-01-07,2024-05-20,499,1411.0,0.065911,0.988119,0.026807,0.507463
7,client_1,hf_content_7,2023-01-08,2024-05-21,499,5775.0,0.019048,0.988119,0.060128,0.524123
8,client_0,hf_content_8,2023-01-09,2024-05-20,497,3498.0,0.044025,0.984158,0.080565,0.532362
9,client_1,hf_content_9,2023-01-10,2024-05-21,497,8099.0,0.017410,0.984158,0.058620,0.521389


In [10]:
# --- 5. Rank Content Items ---
df_merged['rank'] = df_merged['baseline_action_score'].rank(ascending=False)

# --- 6. Assign Reason Codes ---
# Define thresholds for reason codes
STALE_THRESHOLD_DAYS = 200 # Content older than 200 days is considered stale
LOW_CTR_OPPORTUNITY_THRESHOLD = 0.01 # A low CTR indicates opportunity if other conditions met
AVG_POSITION_GOOD_THRESHOLD = 15 # Position better than 15 is considered 'reasonable'
IMPRESSIONS_THRESHOLD = 1000 # Minimum impressions for CTR opportunity to be relevant

def assign_reason_code(row):
    is_stale = row['days_since_update'] > STALE_THRESHOLD_DAYS
    is_low_ctr = row['CTR'] < LOW_CTR_OPPORTUNITY_THRESHOLD
    has_impressions = row['total_impressions'] >= IMPRESSIONS_THRESHOLD
    has_reasonable_pos = row['avg_position'] <= AVG_POSITION_GOOD_THRESHOLD

    if is_stale and is_low_ctr and has_impressions and has_reasonable_pos:
        return 'STALE_AND_LOW_CTR'
    elif is_stale:
        return 'STALE_CONTENT'
    elif is_low_ctr and has_impressions and has_reasonable_pos:
        return 'LOW_CTR_OPPORTUNITY'
    else:
        return 'NONE_APPLICABLE'

df_merged['reason_code'] = df_merged.apply(assign_reason_code, axis=1)

# Sort by rank for the final queue
df_ranked_queue = df_merged.sort_values(by='rank', ascending=True).reset_index(drop=True)

print("Ranked Queue with Reason Codes (Top 10):")
display(df_ranked_queue[['client_hash_id', 'content_hash_id', 'baseline_action_score', 'rank', 'reason_code', 'days_since_update', 'CTR', 'total_impressions', 'avg_position']].head(10))

Ranked Queue with Reason Codes (Top 10):


,client_hash_id,content_hash_id,baseline_action_score,rank,reason_code,days_since_update,CTR,total_impressions,avg_position
0,client_0,hf_content_4,0.708999,1.0,STALE_CONTENT,501,0.013394,9183.0,1.544187
1,client_1,hf_content_81,0.703766,2.0,STALE_CONTENT,425,0.025472,15468.0,11.199480
2,client_0,hf_content_2,0.687001,3.0,STALE_CONTENT,504,0.029532,9752.0,8.972813
3,client_1,hf_content_25,0.684367,4.0,STALE_CONTENT,480,0.026831,12299.0,13.852584
4,client_0,hf_content_36,0.678808,5.0,STALE_CONTENT,470,0.022882,20584.0,27.843805
5,client_1,hf_content_23,0.655614,6.0,STALE_AND_LOW_CTR,483,0.003076,8128.0,4.834240
6,client_0,hf_content_26,0.643327,7.0,STALE_CONTENT,480,0.011254,13151.0,23.125150
7,client_0,hf_content_96,0.632848,8.0,STALE_CONTENT,410,0.011355,20608.0,26.774890
8,client_1,hf_content_59,0.632031,9.0,STALE_CONTENT,447,0.025972,13322.0,19.709590
9,client_1,hf_content_65,0.627073,10.0,STALE_CONTENT,440,0.020560,13181.0,19.245698


In [11]:
# --- 7. Write to CSV ---
import os

output_dir = 'work/outputs'
os.makedirs(output_dir, exist_ok=True)

output_filepath = os.path.join(output_dir, 'baseline_action_score.csv')
df_ranked_queue.to_csv(output_filepath, index=False)

print(f"Ranked queue saved to: {output_filepath}")

Ranked queue saved to: work/outputs/baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [12]:
top_20_review = df_ranked_queue.head(20)

print("--- Top 20 Content Review ---")
print("\nThis review provides an overview of the top 20 ranked content items, including suggested actions, the assigned reason code, a confidence note based on the action score, and potential factors that could make the recommendation inaccurate.")

for index, row in top_20_review.iterrows():
    action = ""
    confidence_note = ""
    what_makes_it_wrong = []

    # Determine Action based on reason code
    if row['reason_code'] == 'STALE_AND_LOW_CTR':
        action = "Comprehensive review: update content for freshness and optimize for CTR."
        confidence_note = "High confidence due to both staleness and low CTR opportunity."
        what_makes_it_wrong.append("The 'content_updated_date' might be incorrect or the content is intentionally evergreen and does not require frequent updates. CTR might be acceptable for the content's purpose.")
    elif row['reason_code'] == 'STALE_CONTENT':
        action = "Review content for freshness and update if necessary."
        confidence_note = "Medium confidence due to significant staleness."
        what_makes_it_wrong.append("The 'content_updated_date' might be incorrect, or the content is intentionally evergreen. The staleness threshold might be too aggressive.")
    elif row['reason_code'] == 'LOW_CTR_OPPORTUNITY':
        action = "Analyze content for improved title/meta description, keyword optimization, or new call-to-actions."
        confidence_note = "Medium confidence due to identified CTR opportunity with reasonable impressions and position."
        what_makes_it_wrong.append("Performance data ('gsc_impressions', 'gsc_clicks') might be incomplete or inaccurate. Low CTR might be acceptable for this specific content type or audience.")
    else:
        action = "No specific action identified by the current rules. Further manual review might be needed."
        confidence_note = "Lower confidence as it didn't trigger primary action rules."
        what_makes_it_wrong.append("The thresholds for reason codes might be too strict, or there are other uncaptured signals indicating a need for action.")

    # General potential issues
    if row['total_impressions'] < IMPRESSIONS_THRESHOLD:
        what_makes_it_wrong.append("Low total impressions: CTR opportunity might be less impactful due to low visibility.")
    if row['gsc_data_available'] == False:
        what_makes_it_wrong.append("GSC data not available: Performance metrics might be missing or inaccurate.")
    if row['is_deleted'] == True or row['is_published'] == False:
        what_makes_it_wrong.append("Content is deleted or unpublished: Action might not be relevant.")
    if row['latest_report_date'].date() < pd.Timestamp.now().date() - pd.Timedelta(days=7):
        what_makes_it_wrong.append("Performance data is not recent; recent changes might not be reflected.")


    print(f"\n--- Item Rank {int(row['rank'])} ---")
    print(f"Content ID: {row['content_hash_id']}")
    print(f"Client ID: {row['client_hash_id']}")
    print(f"Baseline Action Score: {row['baseline_action_score']:.4f}")
    print(f"Reason Code: {row['reason_code']}")
    print(f"Suggested Action: {action}")
    print(f"Confidence Note: {confidence_note}")
    print("What might make this recommendation wrong:")
    for issue in what_makes_it_wrong:
        print(f"  - {issue}")
    print(f"  - General: Underlying data (content dates, performance) may be inaccurate or out of date.")
    print(f"  - General: The thresholds for 'staleness' or 'low CTR opportunity' may not align with current business strategy.")
    print(f"  - General: Content might serve a specific niche or internal purpose where observed metrics are expected.")





--- Top 20 Content Review ---

This review provides an overview of the top 20 ranked content items, including suggested actions, the assigned reason code, a confidence note based on the action score, and potential factors that could make the recommendation inaccurate.

--- Item Rank 1 ---
Content ID: hf_content_4
Client ID: client_0
Baseline Action Score: 0.7090
Reason Code: STALE_CONTENT
Suggested Action: Review content for freshness and update if necessary.
Confidence Note: Medium confidence due to significant staleness.
What might make this recommendation wrong:
  - The 'content_updated_date' might be incorrect, or the content is intentionally evergreen. The staleness threshold might be too aggressive.
  - Performance data is not recent; recent changes might not be reflected.
  - General: Underlying data (content dates, performance) may be inaccurate or out of date.
  - General: The thresholds for 'staleness' or 'low CTR opportunity' may not align with current business strategy.
  -

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak Picks and Leakage Analysis

This section aims to identify entries in the ranked queue that might be considered 'weak picks' due to unreliable data or specific characteristics, and to check for any potential data leakage (e.g., future dates or inappropriate product flags).

We will examine:
1.  **Lack of GSC Data**: Content items for which Google Search Console (GSC) data is not available, which can reduce the confidence in their CTR opportunity scores.
2.  **Old Performance Data**: Items where the latest performance data is not recent, implying that the scores might not reflect current reality.
3.  **Future Dates**: Check if any `content_updated_date` or `latest_report_date` appears to be in the future, indicating potential data leakage or errors.
4.  **Deleted/Unpublished Content**: Items that are marked as deleted or unpublished, as actions on these might be irrelevant.


In [13]:
print("--- Weak Picks Analysis ---")

# 1. Content with no GSC data available
no_gsc_data = df_ranked_queue[df_ranked_queue['gsc_data_available'] == False]
if not no_gsc_data.empty:
    print(f"\nContent without GSC data ({len(no_gsc_data)} items):\n")
    display(no_gsc_data[['client_hash_id', 'content_hash_id', 'rank', 'baseline_action_score', 'reason_code', 'gsc_data_available']].head())
else:
    print("\nAll content in the ranked queue has GSC data available.")

# 2. Content with old performance data (e.g., older than 30 days relative to current date)
DAYS_OLD_PERFORMANCE_THRESHOLD = 30
current_date = pd.Timestamp.now()
old_performance_data = df_ranked_queue[
    (current_date - df_ranked_queue['latest_report_date']).dt.days > DAYS_OLD_PERFORMANCE_THRESHOLD
]

if not old_performance_data.empty:
    print(f"\nContent with performance data older than {DAYS_OLD_PERFORMANCE_THRESHOLD} days ({len(old_performance_data)} items):\n")
    display(old_performance_data[['client_hash_id', 'content_hash_id', 'rank', 'baseline_action_score', 'reason_code', 'latest_report_date']].head())
else:
    print(f"\nAll content in the ranked queue has performance data within the last {DAYS_OLD_PERFORMANCE_THRESHOLD} days.")

print("\n--- Leakage Check ---")

# 3. Check for future dates
future_content_updates = df_ranked_queue[df_ranked_queue['content_updated_date'] > current_date]
future_report_dates = df_ranked_queue[df_ranked_queue['latest_report_date'] > current_date]

if not future_content_updates.empty:
    print(f"\nWARNING: {len(future_content_updates)} items have a 'content_updated_date' in the future:\n")
    display(future_content_updates[['client_hash_id', 'content_hash_id', 'content_updated_date']].head())
else:
    print("\nNo 'content_updated_date' in the future detected.")

if not future_report_dates.empty:
    print(f"\nWARNING: {len(future_report_dates)} items have a 'latest_report_date' in the future:\n")
    display(future_report_dates[['client_hash_id', 'content_hash_id', 'latest_report_date']].head())
else:
    print("\nNo 'latest_report_date' in the future detected.")

# 4. Check for deleted or unpublished content in the ranked queue
deleted_or_unpublished = df_ranked_queue[
    (df_ranked_queue['is_deleted'] == True) |
    (df_ranked_queue['is_published'] == False)
]

if not deleted_or_unpublished.empty:
    print(f"\nContent in the ranked queue that is deleted or unpublished ({len(deleted_or_unpublished)} items):\n")
    display(deleted_or_unpublished[['client_hash_id', 'content_hash_id', 'rank', 'is_deleted', 'is_published']].head())
else:
    print("\nNo deleted or unpublished content found in the ranked queue.")


--- Weak Picks Analysis ---

Content without GSC data (32 items):



,client_hash_id,content_hash_id,rank,baseline_action_score,reason_code,gsc_data_available
68,client_1,hf_content_51,84.5,0.0,NONE_APPLICABLE,False
69,client_1,hf_content_49,84.5,0.0,NONE_APPLICABLE,False
70,client_0,hf_content_52,84.5,0.0,NONE_APPLICABLE,False
71,client_0,hf_content_46,84.5,0.0,NONE_APPLICABLE,False
72,client_1,hf_content_63,84.5,0.0,NONE_APPLICABLE,False



Content with performance data older than 30 days (100 items):



,client_hash_id,content_hash_id,rank,baseline_action_score,reason_code,latest_report_date
0,client_0,hf_content_4,1.0,0.708999,STALE_CONTENT,2024-05-20
1,client_1,hf_content_81,2.0,0.703766,STALE_CONTENT,2024-05-21
2,client_0,hf_content_2,3.0,0.687001,STALE_CONTENT,2024-05-21
3,client_1,hf_content_25,4.0,0.684367,STALE_CONTENT,2024-05-20
4,client_0,hf_content_36,5.0,0.678808,STALE_CONTENT,2024-05-21



--- Leakage Check ---

No 'content_updated_date' in the future detected.

No 'latest_report_date' in the future detected.

No deleted or unpublished content found in the ranked queue.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.